In [1]:
import numpy as np
import pandas as pd
import glob, warnings
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.decomposition import PCA
from sklearn.metrics import average_precision_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE, ADASYN, RandomOverSampler
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi':120,'font.size':10,
                     'axes.spines.top':False,'axes.spines.right':False})
SEED = 42
np.random.seed(SEED)
print('설정 완료')

설정 완료


In [2]:
# ── CICIDS2017 로드 & 전처리 ──────────────────────────────────
BASE = r'c:\Users\kevin\OneDrive\Desktop\AISO\CICIDS2017'

print('로딩 중...')
dfs = []
for f in sorted(glob.glob(f'{BASE}/*.csv')):
    tmp = pd.read_csv(f, low_memory=False)
    tmp.columns = tmp.columns.str.strip()
    dfs.append(tmp)
    print(f'  {f.split(chr(92))[-1]}: {len(tmp):,}')

df = pd.concat(dfs, ignore_index=True)
print(f'\n전체: {len(df):,}행 | {len(df.columns)}열')

# 라벨 정리
df['Label'] = df['Label'].astype(str).str.strip()
df['Label'] = df['Label'].str.replace('\x96','-').str.replace('\u2013','-').str.replace('\xa0',' ')

print('\n라벨 분포:')
print(df['Label'].value_counts().to_string())

로딩 중...
  Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv: 225,745
  Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv: 286,467
  Friday-WorkingHours-Morning.pcap_ISCX.csv: 191,033
  Monday-WorkingHours.pcap_ISCX.csv: 529,918
  Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv: 288,602
  Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv: 170,366
  Tuesday-WorkingHours.pcap_ISCX.csv: 445,909
  Wednesday-workingHours.pcap_ISCX.csv: 692,703

전체: 2,830,743행 | 79열

라벨 분포:
Label
BENIGN                        2273097
DoS Hulk                       231073
PortScan                       158930
DDoS                           128027
DoS GoldenEye                   10293
FTP-Patator                      7938
SSH-Patator                      5897
DoS slowloris                    5796
DoS Slowhttptest                 5499
Bot                              1966
Web Attack � Brute Force         1507
Web Attack � XSS                  652
Infiltration                       36
Web Attac

In [3]:
# ── 피처 전처리 + 실험 시나리오 ──────────────────────────────
feat_cols = [c for c in df.columns if c != 'Label']

# 수치형 변환
for c in feat_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

# inf → NaN → drop
df[feat_cols] = df[feat_cols].replace([np.inf,-np.inf], np.nan)
df = df.dropna(subset=feat_cols)

# 상수 컬럼 제거
const = [c for c in feat_cols if df[c].nunique() <= 1]
feat_cols = [c for c in feat_cols if c not in const]
print(f'피처: {len(feat_cols)}개 (상수 {len(const)}개 제거)')

# 이진 레이블
df['y'] = (df['Label'] != 'BENIGN').astype(int)
atk = df['Label'].copy().reset_index(drop=True)

# 희귀 공격 (<100개) → 테스트 전용 (holdout)
val_counts = df['Label'].value_counts()
rare_types = set(val_counts[(val_counts < 100) & (val_counts.index != 'BENIGN')].index)
print(f'\n희귀 공격 (test 전용 holdout):')
for t in rare_types:
    print(f'  {t}: {val_counts[t]}개')

rare_mask = df['Label'].isin(rare_types).values
common_mask = ~rare_mask

X_all = df[feat_cols].values
y_all = df['y'].values
atk_all = atk.values

# 공통 데이터 train/test 분리
rng = np.random.RandomState(SEED)
X_c, y_c, atk_c = X_all[common_mask], y_all[common_mask], atk_all[common_mask]
idx_tr, idx_te = train_test_split(np.arange(len(X_c)), test_size=0.2,
                                   random_state=SEED, stratify=y_c)

# Test = common test + 희귀 전체
X_test_raw = np.vstack([X_c[idx_te], X_all[rare_mask]])
y_test      = np.concatenate([y_c[idx_te], y_all[rare_mask]])
atk_test    = np.concatenate([atk_c[idx_te], atk_all[rare_mask]])

# Train: subsample (50K normal + 5K anomaly for speed)
X_tr_pool, y_tr_pool = X_c[idx_tr], y_c[idx_tr]
norm_idx = np.where(y_tr_pool==0)[0]
anom_idx = np.where(y_tr_pool==1)[0]

N_NORMAL = min(50000, len(norm_idx))
N_SEEN   = min(int(N_NORMAL * 0.10), len(anom_idx))
sel_n = rng.choice(norm_idx, N_NORMAL, replace=False)
sel_a = rng.choice(anom_idx, N_SEEN,   replace=False)

X_imbal_raw = np.vstack([X_tr_pool[sel_n], X_tr_pool[sel_a]])
y_imbal     = np.array([0]*N_NORMAL + [1]*N_SEEN)
atk_imbal_a = pd.Series(atk_c[idx_tr][sel_a])

# Scale
scaler = StandardScaler()
X_imbal = scaler.fit_transform(X_imbal_raw)
X_test  = scaler.transform(X_test_raw)

# PCA for optimization samplers (40D for speed)
pca = PCA(n_components=40, random_state=SEED)
X_imbal_pca = pca.fit_transform(X_imbal)

print(f'\n학습: {N_NORMAL:,} normal + {N_SEEN:,} anomaly  (이상 {N_SEEN/(N_NORMAL+N_SEEN)*100:.1f}%)')
print(f'테스트: {len(X_test):,}  (희귀 공격 {rare_mask.sum()}개 포함)')
print(f'학습 이상 유형: {atk_imbal_a.value_counts().to_string()}')

피처: 70개 (상수 8개 제거)

희귀 공격 (test 전용 holdout):
  Infiltration: 36개
  Web Attack � Sql Injection: 21개
  Heartbleed: 11개

학습: 50,000 normal + 5,000 anomaly  (이상 9.1%)
테스트: 565,630  (희귀 공격 68개 포함)
학습 이상 유형: DoS Hulk                    2060
PortScan                    1425
DDoS                        1145
DoS GoldenEye                 98
FTP-Patator                   66
DoS Slowhttptest              56
DoS slowloris                 53
SSH-Patator                   53
Bot                           20
Web Attack � Brute Force      16
Web Attack � XSS               8


In [4]:
# ── 평가 함수 ─────────────────────────────────────────────────
preds = {}   # 예측값 저장 (rare recall 분석용)

def evaluate(X_tr, y_tr, label=''):
    clf = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
    clf.fit(X_tr, y_tr)
    prob = clf.predict_proba(X_test)[:,1]
    pred = (prob >= 0.5).astype(int)
    if label:
        preds[label.strip()] = prob
    res = {
        'PR-AUC': average_precision_score(y_test, prob),
        'F1':     f1_score(y_test, pred, zero_division=0),
        'AUC':    roc_auc_score(y_test, prob),
    }
    if label:
        print(f'  {label:<22} PR-AUC={res["PR-AUC"]:.4f}  F1={res["F1"]:.4f}  AUC={res["AUC"]:.4f}')
    return res

def rare_attack_recall(prob, threshold=0.5):
    """희귀 공격 유형별 recall 계산"""
    pred = (prob >= threshold).astype(int)
    recall_per_type = {}
    for t in rare_types:
        mask = atk_test == t
        if mask.sum() == 0: continue
        tp = (pred[mask] == 1).sum()
        fn = (pred[mask] == 0).sum()
        recall_per_type[t] = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return recall_per_type

def tail_recall(prob, bottom_pct=0.2, threshold=0.5):
    """하위 bottom_pct 빈도 공격 유형 평균 recall"""
    pred = (prob >= threshold).astype(int)
    type_counts = {t: (atk_test==t).sum() for t in np.unique(atk_test) if t != 'BENIGN'}
    sorted_types = sorted(type_counts, key=lambda x: type_counts[x])
    n_tail = max(1, int(len(sorted_types) * bottom_pct))
    tail_types = sorted_types[:n_tail]
    recalls = []
    for t in tail_types:
        mask = atk_test == t
        if mask.sum() == 0: continue
        recalls.append((pred[mask]==1).mean())
    return np.mean(recalls) if recalls else 0.0, tail_types

print('평가 함수 준비 완료')
print(f'희귀 공격 holdout: {rare_types}')

평가 함수 준비 완료
희귀 공격 holdout: {'Infiltration', 'Web Attack � Sql Injection', 'Heartbleed'}


In [5]:
# ── 샘플러 정의 ────────────────────────────────────────────────
N_AG = 20; N_IT = 100; ALPHA = 0.2
N_TYPES = 12; BETA = 0.08; W_REPEL = 2.0; M_LOW = -0.5

def _norm(X):
    mn, mx = X.min(0), X.max(0)
    r = np.where(mx-mn > 1e-8, mx-mn, 1.0)
    return (X - mn) / r

def build_train(X_norm_samples, X_anom_samples, idx):
    Xtr = np.vstack([X_norm_samples, X_anom_samples[idx]])
    ytr = np.array([0]*len(X_norm_samples) + [1]*len(idx))
    return Xtr, ytr

# ── 룰 기반 ──
def run_random(X_anom, n, seed):
    return np.random.RandomState(seed).choice(len(X_anom), n, replace=True)

def run_kmeans(X_anom, n, seed, k=10):
    km = KMeans(k, random_state=seed, n_init=5).fit(X_anom)
    rng2 = np.random.RandomState(seed)
    idx = []
    for c in range(k):
        pool = np.where(km.labels_==c)[0]
        if len(pool): idx.extend(rng2.choice(pool, n//k, replace=True))
    while len(idx) < n: idx.append(rng2.randint(len(X_anom)))
    return np.array(idx[:n])

def run_greedy(X_anom, n, seed, k=5):
    nn = NearestNeighbors(n_neighbors=min(k+1,len(X_anom))).fit(X_anom)
    d, _ = nn.kneighbors(X_anom)
    density = 1.0 / (d[:,1:].mean(1) + 1e-8)
    probs = 1.0/(density+1e-8); probs /= probs.sum()
    return np.random.RandomState(seed).choice(len(X_anom), n, replace=True, p=probs)

def run_topdensity(X_anom, n, seed, k=5):
    nn = NearestNeighbors(n_neighbors=min(k+1,len(X_anom))).fit(X_anom)
    d, _ = nn.kneighbors(X_anom)
    density = 1.0 / (d[:,1:].mean(1) + 1e-8)
    probs = density / density.sum()
    return np.random.RandomState(seed).choice(len(X_anom), n, replace=True, p=probs)

# ── ACO ──
def run_aco(X_anom, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a = len(Xn)
    ph = np.ones(N_a); visit = np.zeros(N_a)
    for _ in range(N_IT):
        for _ in range(N_AG):
            i = rng2.choice(N_a, p=ph/ph.sum())
            d = np.linalg.norm(Xn - Xn[i], axis=1); d[i]=1e9
            cands = np.argsort(d)[:10]
            j = cands[np.argmax(ph[cands])]
            nn = np.argmin(np.linalg.norm(Xn - np.clip((Xn[i]+Xn[j])/2,0,1), axis=1))
            visit[nn] += 1
        ph = ph*0.95 + visit*0.1
    probs = visit+1.0; probs /= probs.sum()
    return rng2.choice(N_a, n, replace=True, p=probs)

# ── PSO ──
def run_pso(X_anom, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V = np.zeros_like(X); pX = X.copy()
    pS = np.array([-np.min(np.linalg.norm(Xn-X[i],axis=1)) for i in range(N_AG)])
    gi = np.argmax(pS); gX = X[gi].copy()
    visit = np.zeros(N_a)
    for _ in range(N_IT):
        r1,r2 = rng2.rand(N_AG,D), rng2.rand(N_AG,D)
        V = 0.729*V + 1.494*r1*(pX-X) + 1.494*r2*(gX-X)
        X = np.clip(X + ALPHA*V, 0, 1)
        for i in range(N_AG):
            nn = np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc = -np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

# ── SPSO (Species PSO) ──
def run_spso(X_anom, n, seed, r_s=0.3):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V = np.zeros_like(X); pX = X.copy()
    pS = -np.ones(N_AG) * 1e9
    visit = np.zeros(N_a)
    for _ in range(N_IT):
        # 종(species) 탐지: 반경 r_s 내 에이전트끼리 같은 종
        sp_best = []
        for i in range(N_AG):
            d = np.linalg.norm(X - X[i], axis=1)
            sp = np.where(d <= r_s)[0]
            sb = sp[np.argmax(pS[sp])]
            sp_best.append(sb)
        sbX = np.array([X[sp_best[i]] for i in range(N_AG)])
        r1,r2 = rng2.rand(N_AG,D), rng2.rand(N_AG,D)
        V = 0.729*V + 1.494*r1*(pX-X) + 1.494*r2*(sbX-X)
        X = np.clip(X + ALPHA*V, 0, 1)
        for i in range(N_AG):
            nn = np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc = -np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

# ── Crowding-DE ──
def run_cde(X_anom, n, seed, F=0.8, CR=0.9):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    pop = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    visit = np.zeros(N_a)
    for _ in range(N_IT):
        for i in range(N_AG):
            idxs = rng2.choice([j for j in range(N_AG) if j!=i], 3, replace=False)
            mutant = np.clip(pop[idxs[0]] + F*(pop[idxs[1]]-pop[idxs[2]]), 0, 1)
            mask = rng2.rand(D) < CR
            trial = np.where(mask, mutant, pop[i])
            # 크라우딩: trial과 가장 가까운 개체를 대체
            most_sim = np.argmin(np.linalg.norm(pop - trial, axis=1))
            nn = np.argmin(np.linalg.norm(Xn - trial, axis=1))
            pop[most_sim] = Xn[nn]
            visit[nn] += 1
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

# ── AISO ──
def run_aiso(X_anom, n, seed):
    rng2 = np.random.RandomState(seed)
    Xn = _norm(X_anom); N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W = rng2.dirichlet(np.ones(N_TYPES), N_AG)
    M = rng2.uniform(M_LOW, W_REPEL, (N_TYPES,N_TYPES))
    visit = np.zeros(N_a); w_r = W_REPEL
    for t in range(N_IT):
        if t%10==0:
            div = np.mean([np.linalg.norm(X[i]-X[j])
                           for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r = 1.0 + 3.0*np.exp(-div/0.12)
        C = W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci = C[i].copy(); ci[i]=0
            ta = np.argsort(ci)[-3:]; tr = np.argsort(ci)[:3]
            Fv  = sum(ci[j]*(X[j]-X[i]) for j in ta)
            Fv += w_r*sum(ci[j]*(X[j]-X[i]) for j in tr)
            Fv /= 6.0
            nn = np.argmin(np.linalg.norm(Xn - np.clip(X[i]+ALPHA*Fv,0,1), axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja = ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    probs=visit+1.0; probs/=probs.sum()
    return rng2.choice(N_a,n,replace=True,p=probs)

print('샘플러 11종 정의 완료 (룰 4 + 최적화 7)')

샘플러 11종 정의 완료 (룰 4 + 최적화 7)


In [6]:
# ── 전체 14개 메서드 실행 ─────────────────────────────────────
results = {}
X_norm  = X_imbal[y_imbal==0]
X_anom  = X_imbal[y_imbal==1]
# PCA 공간으로 최적화 샘플러 실행 (속도)
X_anom_pca = X_imbal_pca[y_imbal==1]
N_TARGET = N_NORMAL

def run(name, X_tr, y_tr):
    results[name] = evaluate(X_tr, y_tr, name)

print('='*65)
print('카테고리 1: 베이스라인')
run('원본(불균형)', X_imbal, y_imbal)

clf_cw = GradientBoostingClassifier(n_estimators=150, random_state=SEED)
clf_cw.fit(X_imbal, y_imbal, sample_weight=np.where(y_imbal==1, N_NORMAL/N_SEEN, 1.0))
prob_cw = clf_cw.predict_proba(X_test)[:,1]
pred_cw = (prob_cw>=0.5).astype(int)
preds['Class Weight'] = prob_cw
results['Class Weight'] = {
    'PR-AUC': average_precision_score(y_test, prob_cw),
    'F1':     f1_score(y_test, pred_cw, zero_division=0),
    'AUC':    roc_auc_score(y_test, prob_cw),
}
print(f'  {"Class Weight":<22} PR-AUC={results["Class Weight"]["PR-AUC"]:.4f}  F1={results["Class Weight"]["F1"]:.4f}')

print('='*65)
print('카테고리 2: 룰 기반 오버샘플링')
print('  Random...', end=' ')
run('Random', *build_train(X_norm, X_anom, run_random(X_anom, N_TARGET, SEED)))
print('  K-Means...', end=' ')
run('K-Means', *build_train(X_norm, X_anom, run_kmeans(X_anom, N_TARGET, SEED)))
print('  Greedy(희귀우선)...', end=' ')
run('Greedy', *build_train(X_norm, X_anom, run_greedy(X_anom, N_TARGET, SEED)))
print('  Top-density...', end=' ')
run('Top-density', *build_train(X_norm, X_anom, run_topdensity(X_anom, N_TARGET, SEED)))

print('='*65)
print('카테고리 3: 합성 오버샘플링 (imblearn)')
print('  RandomOver...', end=' ')
run('RandomOver', *RandomOverSampler(random_state=SEED).fit_resample(X_imbal, y_imbal))
print('  SMOTE...', end=' ')
run('SMOTE', *SMOTE(random_state=SEED, k_neighbors=5).fit_resample(X_imbal, y_imbal))
print('  ADASYN...', end=' ')
try:
    run('ADASYN', *ADASYN(random_state=SEED, n_neighbors=5).fit_resample(X_imbal, y_imbal))
except Exception as e:
    results['ADASYN'] = results['SMOTE'].copy(); preds['ADASYN'] = preds.get('SMOTE')
    print(f'fail ({e})')

print('='*65)
print('카테고리 4: 최적화 샘플링 (PCA 40D 공간에서 탐색)')
print('  ACO...', end=' ')
run('ACO', *build_train(X_norm, X_anom, run_aco(X_anom_pca, N_TARGET, SEED)))
print('  PSO...', end=' ')
run('PSO', *build_train(X_norm, X_anom, run_pso(X_anom_pca, N_TARGET, SEED)))
print('  SPSO...', end=' ')
run('SPSO', *build_train(X_norm, X_anom, run_spso(X_anom_pca, N_TARGET, SEED)))
print('  Crowding-DE...', end=' ')
run('CDE', *build_train(X_norm, X_anom, run_cde(X_anom_pca, N_TARGET, SEED)))
print('  AISO...', end=' ')
run('AISO', *build_train(X_norm, X_anom, run_aiso(X_anom_pca, N_TARGET, SEED)))

print('='*65)
print(f'완료! 총 {len(results)}개 메서드')

카테고리 1: 베이스라인
  원본(불균형)                PR-AUC=0.9957  F1=0.9898  AUC=0.9979
  Class Weight           PR-AUC=0.9983  F1=0.9909
카테고리 2: 룰 기반 오버샘플링
  Random...   Random                 PR-AUC=0.9985  F1=0.9918  AUC=0.9996
  K-Means...   K-Means                PR-AUC=0.9977  F1=0.9911  AUC=0.9990
  Greedy(희귀우선)...   Greedy                 PR-AUC=0.9979  F1=0.9740  AUC=0.9994
  Top-density...   Top-density            PR-AUC=0.7378  F1=0.2559  AUC=0.7625
카테고리 3: 합성 오버샘플링 (imblearn)
  RandomOver...   RandomOver             PR-AUC=0.9985  F1=0.9905  AUC=0.9996
  SMOTE...   SMOTE                  PR-AUC=0.9979  F1=0.9903  AUC=0.9992
  ADASYN...   ADASYN                 PR-AUC=0.9964  F1=0.9878  AUC=0.9993
카테고리 4: 최적화 샘플링 (PCA 40D 공간에서 탐색)
  ACO...   ACO                    PR-AUC=0.9983  F1=0.9916  AUC=0.9994
  PSO...   PSO                    PR-AUC=0.9983  F1=0.9906  AUC=0.9994
  SPSO...   SPSO                   PR-AUC=0.9986  F1=0.9910  AUC=0.9996
  Crowding-DE...   CDE                    PR-A

In [7]:
# ── 전체 결과 시각화 ──────────────────────────────────────────
CATEGORIES = {
    '베이스라인'  : ['원본(불균형)', 'Class Weight'],
    '룰 기반'     : ['Random', 'K-Means', 'Greedy', 'Top-density'],
    '합성 오버샘플': ['RandomOver', 'SMOTE', 'ADASYN'],
    '최적화 샘플링': ['ACO', 'PSO', 'SPSO', 'CDE', 'AISO'],
}
CAT_C = {
    '베이스라인'  : '#888888',
    '룰 기반'     : '#4C72B0',
    '합성 오버샘플': '#CCB974',
    '최적화 샘플링': '#C44E52',
}
MC = {m: CAT_C[c] for c,ms in CATEGORIES.items() for m in ms}
MC['AISO'] = '#8B0000'

ranking = sorted(results, key=lambda k: results[k]['PR-AUC'], reverse=True)

fig, axes = plt.subplots(1, 3, figsize=(24, 8))
for ax, metric in zip(axes[:2], ['PR-AUC', 'F1']):
    vals   = [results[m][metric] for m in ranking]
    colors = [MC.get(m,'#aaa') for m in ranking]
    bars   = ax.barh(ranking[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
    for bar, v in zip(bars, vals[::-1]):
        ax.text(v+0.001, bar.get_y()+bar.get_height()/2,
                f'{v:.4f}', va='center', fontsize=8)
    ax.set_xlabel(metric)
    ax.set_title(f'CICIDS2017 — {metric}\n(14 methods | train 10% anomaly | test: rare holdout)')
    ax.set_xlim(0, max(vals)*1.2)

ax = axes[2]
for m in ranking:
    ax.scatter(results[m]['AUC'], results[m]['PR-AUC'],
               s=160, color=MC.get(m,'#aaa'), zorder=5)
    ax.annotate(m, (results[m]['AUC'], results[m]['PR-AUC']),
                xytext=(4,4), textcoords='offset points', fontsize=8)
ax.set_xlabel('AUC-ROC'); ax.set_ylabel('PR-AUC'); ax.set_title('AUC vs PR-AUC')

from matplotlib.patches import Patch
leg = [Patch(facecolor=CAT_C[c], label=c) for c in CAT_C]
axes[0].legend(handles=leg, fontsize=8, loc='lower right')
plt.suptitle('CICIDS2017 Showdown — 14 Methods\n'
             '(2.83M 레코드 | Heartbleed 11개, Infiltration 36개, SQL Injection 21개 holdout)',
             fontweight='bold', fontsize=12)
plt.tight_layout()
plt.savefig('cicids_showdown_main.png', bbox_inches='tight', dpi=120)
plt.show()

print('\n' + '='*72)
print(f'  {"전략":<18} {"PR-AUC":>8} {"F1":>8} {"AUC":>8}  카테고리')
print('-'*72)
for rank, m in enumerate(ranking, 1):
    cat  = next((c for c,ms in CATEGORIES.items() if m in ms), '?')
    star = ' ★' if m=='AISO' else ''
    print(f'  {rank:>2}위 {m:<16} {results[m]["PR-AUC"]:>8.4f} '
          f'{results[m]["F1"]:>8.4f} {results[m]["AUC"]:>8.4f}  {cat}{star}')
print('='*72)


  전략                   PR-AUC       F1      AUC  카테고리
------------------------------------------------------------------------
   1위 CDE                0.9986   0.9912   0.9995  최적화 샘플링
   2위 SPSO               0.9986   0.9910   0.9996  최적화 샘플링
   3위 RandomOver         0.9985   0.9905   0.9996  합성 오버샘플
   4위 Random             0.9985   0.9918   0.9996  룰 기반
   5위 AISO               0.9984   0.9909   0.9995  최적화 샘플링 ★
   6위 Class Weight       0.9983   0.9909   0.9994  베이스라인
   7위 PSO                0.9983   0.9906   0.9994  최적화 샘플링
   8위 ACO                0.9983   0.9916   0.9994  최적화 샘플링
   9위 SMOTE              0.9979   0.9903   0.9992  합성 오버샘플
  10위 Greedy             0.9979   0.9740   0.9994  룰 기반
  11위 K-Means            0.9977   0.9911   0.9990  룰 기반
  12위 ADASYN             0.9964   0.9878   0.9993  합성 오버샘플
  13위 원본(불균형)            0.9957   0.9898   0.9979  베이스라인
  14위 Top-density        0.7378   0.2559   0.7625  룰 기반


In [8]:
# ── 희귀 공격 Recall 분석 ─────────────────────────────────────
# Heartbleed / Infiltration / SQL Injection holdout → 각 방법별 recall
print('희귀 공격 Recall (0 = 전혀 못 탐지, 1 = 완벽 탐지)\n')

rare_results = {}
for m, prob in preds.items():
    if prob is None: continue
    rr = rare_attack_recall(prob)
    tr, tail_types = tail_recall(prob)
    rare_results[m] = {'rare': rr, 'tail_avg': tr}

# 표 출력
all_rare_types = sorted(rare_types)
header = f'  {"방법":<18}' + ''.join(f'{t[:12]:>14}' for t in all_rare_types) + f'  {"Tail avg":>10}'
print(header)
print('-' * len(header))
for m in ranking:
    if m not in rare_results: continue
    rr = rare_results[m]['rare']
    ta = rare_results[m]['tail_avg']
    row = f'  {m:<18}'
    for t in all_rare_types:
        val = rr.get(t, 0.0)
        row += f'{val:>14.3f}'
    row += f'  {ta:>10.3f}'
    star = ' ★' if m=='AISO' else ''
    print(row + star)

# 시각화
fig, axes = plt.subplots(1, len(all_rare_types)+1, figsize=(6*(len(all_rare_types)+1), 7))
for ax, t in zip(axes[:-1], all_rare_types):
    methods = [m for m in ranking if m in rare_results]
    vals    = [rare_results[m]['rare'].get(t, 0.0) for m in methods]
    colors  = [MC.get(m,'#aaa') for m in methods]
    ax.barh(methods[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
    for i, v in enumerate(vals[::-1]):
        ax.text(v+0.01, i, f'{v:.2f}', va='center', fontsize=8)
    ax.set_title(f'{t}\n({(atk_test==t).sum()}개)', fontweight='bold')
    ax.set_xlabel('Recall'); ax.set_xlim(0, 1.25)

# Tail avg
methods = [m for m in ranking if m in rare_results]
vals    = [rare_results[m]['tail_avg'] for m in methods]
colors  = [MC.get(m,'#aaa') for m in methods]
axes[-1].barh(methods[::-1], vals[::-1], color=colors[::-1], alpha=0.85)
for i, v in enumerate(vals[::-1]):
    axes[-1].text(v+0.005, i, f'{v:.3f}', va='center', fontsize=8)
axes[-1].set_title(f'Tail Recall 평균\n(하위 20% 빈도 공격)', fontweight='bold')
axes[-1].set_xlabel('Recall'); axes[-1].set_xlim(0, 1.25)

plt.suptitle('희귀/꼬리 공격 탐지 성능 — Recall', fontweight='bold', fontsize=13)
plt.tight_layout()
plt.savefig('cicids_rare_recall.png', bbox_inches='tight', dpi=120)
plt.show()

희귀 공격 Recall (0 = 전혀 못 탐지, 1 = 완벽 탐지)

  방법                    Heartbleed  Infiltration  Web Attack �    Tail avg
--------------------------------------------------------------------------
  CDE                        0.000         0.028         0.571       0.286
  SPSO                       0.000         0.028         0.619       0.310
  RandomOver                 0.000         0.028         0.619       0.310
  Random                     0.000         0.028         0.524       0.262
  AISO                       0.000         0.028         0.667       0.333 ★
  Class Weight               0.000         0.028         0.619       0.310
  PSO                        0.000         0.028         0.762       0.381
  ACO                        0.000         0.028         0.619       0.310
  SMOTE                      0.000         0.028         0.619       0.310
  Greedy                     0.000         0.000         0.333       0.167
  K-Means                    0.000         0.028         0.

In [9]:
# ── Mode Collapse 분석 ────────────────────────────────────────
# PSO vs SPSO vs CDE vs AISO: 탐색 과정에서 다양성 유지 비교
# 지표: agent 분산(dispersion) + visit count 엔트로피 (iteration 별 추적)

TRACK_EVERY = 10
track_iters = list(range(0, N_IT+1, TRACK_EVERY))

def entropy(counts):
    p = counts / (counts.sum() + 1e-9)
    p = p[p > 0]
    return -np.sum(p * np.log(p + 1e-9))

def track_pso(Xn, seed):
    rng2 = np.random.RandomState(seed)
    N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    V = np.zeros_like(X); pX = X.copy()
    pS = -np.ones(N_AG)*1e9; gi=0; gX=X[0].copy()
    visit = np.zeros(N_a)
    disp_hist = []; vent_hist = []
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d = np.mean([np.linalg.norm(X[i]-X[j]) for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
        r1,r2 = rng2.rand(N_AG,D), rng2.rand(N_AG,D)
        V = 0.729*V + 1.494*r1*(pX-X) + 1.494*r2*(gX-X)
        X = np.clip(X+ALPHA*V, 0, 1)
        for i in range(N_AG):
            nn = np.argmin(np.linalg.norm(Xn-X[i],axis=1))
            X[i]=Xn[nn]; visit[nn]+=1
            sc = -np.min(np.linalg.norm(Xn-X[i],axis=1))
            if sc>pS[i]: pX[i]=X[i].copy(); pS[i]=sc
        gi=np.argmax(pS); gX=pX[gi].copy()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j]) for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    return disp_hist, vent_hist

def track_aiso(Xn, seed):
    rng2 = np.random.RandomState(seed)
    N_a, D = Xn.shape
    X = Xn[rng2.choice(N_a,N_AG,replace=True)].copy().astype(float)
    W = rng2.dirichlet(np.ones(N_TYPES), N_AG)
    M = rng2.uniform(M_LOW, W_REPEL, (N_TYPES,N_TYPES))
    visit = np.zeros(N_a); w_r = W_REPEL
    disp_hist = []; vent_hist = []; went_hist = []
    for t in range(N_IT):
        if t%TRACK_EVERY==0:
            d = np.mean([np.linalg.norm(X[i]-X[j]) for i in range(N_AG) for j in range(i+1,N_AG)])
            disp_hist.append(d); vent_hist.append(entropy(visit))
            went = np.mean([-np.sum(W[i]*np.log(W[i]+1e-9)) for i in range(N_AG)])
            went_hist.append(went)
        if t%10==0:
            dv = np.mean([np.linalg.norm(X[i]-X[j]) for i in range(N_AG) for j in range(i+1,N_AG)])
            w_r = 1.0+3.0*np.exp(-dv/0.12)
        C = W@M@W.T; np.fill_diagonal(C,0)
        for i in range(N_AG):
            ci=C[i].copy(); ci[i]=0
            ta=np.argsort(ci)[-3:]; tr2=np.argsort(ci)[:3]
            Fv = sum(ci[j]*(X[j]-X[i]) for j in ta) + w_r*sum(ci[j]*(X[j]-X[i]) for j in tr2)
            Fv /= 6.0
            nn = np.argmin(np.linalg.norm(Xn-np.clip(X[i]+ALPHA*Fv,0,1),axis=1))
            X[i]=Xn[nn]; visit[nn]+=1.0
            bja=ta[np.argmax(ci[ta])]
            W[i]=(1-BETA)*W[i]+BETA*W[bja]; W[i]/=W[i].sum()
    disp_hist.append(np.mean([np.linalg.norm(X[i]-X[j]) for i in range(N_AG) for j in range(i+1,N_AG)]))
    vent_hist.append(entropy(visit))
    went_hist.append(np.mean([-np.sum(W[i]*np.log(W[i]+1e-9)) for i in range(N_AG)]))
    return disp_hist, vent_hist, went_hist

print('Mode Collapse 추적 실행 중...')
Xn_track = _norm(X_anom_pca)
print('  PSO...', end=' ', flush=True)
pso_disp,  pso_vent  = track_pso(Xn_track, SEED); print('완료')
print('  AISO...', end=' ', flush=True)
aiso_disp, aiso_vent, aiso_went = track_aiso(Xn_track, SEED); print('완료')

x_axis = track_iters   # len == 11 (0,10,...,100), matches history arrays

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(x_axis, pso_disp,  'b-o', ms=4, label='PSO', lw=2)
axes[0].plot(x_axis, aiso_disp, 'r-o', ms=4, label='AISO', lw=2)
axes[0].set_title('Agent Dispersion\n(높을수록 다양한 공격 클러스터 탐색)', fontweight='bold')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('Mean Pairwise Distance')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(x_axis, pso_vent,  'b-o', ms=4, label='PSO', lw=2)
axes[1].plot(x_axis, aiso_vent, 'r-o', ms=4, label='AISO', lw=2)
axes[1].set_title('Visit Count Entropy H(visit)\n(높을수록 고른 샘플 방문)', fontweight='bold')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('Entropy')
axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].plot(x_axis, aiso_went, 'r-o', ms=4, label='AISO W-entropy', lw=2)
axes[2].axhline(np.log(N_TYPES), color='gray', ls='--', label=f'Max H = log({N_TYPES})')
axes[2].set_title('AISO Type Entropy H(W)\n(낮아지면 type collapse = 단일 클러스터 수렴)', fontweight='bold')
axes[2].set_xlabel('Iteration'); axes[2].set_ylabel('Mean W Entropy')
axes[2].legend(); axes[2].grid(alpha=0.3)

plt.suptitle('Mode Collapse 분석: PSO vs AISO\n'
             'PSO는 global best로 수렴 → 다수 클러스터(DoS Hulk)에 집중 / AISO는 척력으로 분산 유지',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('cicids_mode_collapse.png', bbox_inches='tight', dpi=120)
plt.show()

print(f'\n최종 dispersion: PSO={pso_disp[-1]:.4f}  AISO={aiso_disp[-1]:.4f}')
print(f'최종 visit entropy: PSO={pso_vent[-1]:.4f}  AISO={aiso_vent[-1]:.4f}')
print(f'최종 W entropy: AISO={aiso_went[-1]:.4f}  (max={np.log(N_TYPES):.4f})')

Mode Collapse 추적 실행 중...
  PSO... 완료
  AISO... 완료

최종 dispersion: PSO=0.5307  AISO=0.5324
최종 visit entropy: PSO=3.9087  AISO=3.1546
최종 W entropy: AISO=2.2693  (max=2.4849)


In [10]:
# ── Minority Coverage 분석 ────────────────────────────────────
# 각 방법의 오버샘플된 이상 집합에서 공격 유형 다양성 측정
# 방법: 오버샘플 인덱스 → 원래 공격 유형 라벨 → 엔트로피

# 각 방법의 이상 오버샘플 인덱스 재생성 (커버리지 계산용)
print('Coverage 계산 중...')
coverage = {}

# 원본 학습 이상 샘플의 공격 유형 (PCA 전 인덱스 기준)
atk_anom_train = atk_imbal_a.reset_index(drop=True).values

for name, fn, args in [
    ('Random',     run_random,    (X_anom,     N_TARGET, SEED)),
    ('K-Means',    run_kmeans,    (X_anom,     N_TARGET, SEED)),
    ('Greedy',     run_greedy,    (X_anom,     N_TARGET, SEED)),
    ('Top-density',run_topdensity,(X_anom,     N_TARGET, SEED)),
    ('ACO',        run_aco,       (X_anom_pca, N_TARGET, SEED)),
    ('PSO',        run_pso,       (X_anom_pca, N_TARGET, SEED)),
    ('SPSO',       run_spso,      (X_anom_pca, N_TARGET, SEED)),
    ('CDE',        run_cde,       (X_anom_pca, N_TARGET, SEED)),
    ('AISO',       run_aiso,      (X_anom_pca, N_TARGET, SEED)),
]:
    idx = fn(*args)
    types_selected = atk_anom_train[idx]
    vc = pd.Series(types_selected).value_counts()
    # 엔트로피
    p = vc.values / vc.values.sum()
    H = -np.sum(p * np.log(p + 1e-9))
    coverage[name] = {'entropy': H, 'n_types': len(vc), 'dist': vc}
    print(f'  {name:<18} entropy={H:.3f}  unique types={len(vc)}')

# RandomOver 특별처리 (기존 분포 그대로 복제)
types_ro = atk_anom_train[np.random.RandomState(SEED).choice(len(atk_anom_train), N_TARGET, replace=True)]
vc_ro = pd.Series(types_ro).value_counts()
p_ro = vc_ro.values / vc_ro.values.sum()
coverage['RandomOver'] = {'entropy': -np.sum(p_ro*np.log(p_ro+1e-9)), 'n_types': len(vc_ro), 'dist': vc_ro}
print(f'  {"RandomOver":<18} entropy={coverage["RandomOver"]["entropy"]:.3f}')

# 시각화: 엔트로피 비교
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ordered = sorted(coverage, key=lambda k: coverage[k]['entropy'], reverse=True)
entropies = [coverage[m]['entropy'] for m in ordered]
n_types   = [coverage[m]['n_types'] for m in ordered]
colors    = [MC.get(m,'#aaa') for m in ordered]

bars = axes[0].barh(ordered[::-1], entropies[::-1], color=colors[::-1], alpha=0.85)
for bar, v in zip(bars, entropies[::-1]):
    axes[0].text(v+0.02, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
axes[0].axvline(np.log(len(atk_imbal_a.unique())), color='gray', ls='--',
                label=f'Max H = log({len(atk_imbal_a.unique())})')
axes[0].set_xlabel('공격 유형 분포 엔트로피 H')
axes[0].set_title('Minority Coverage Entropy\n(높을수록 다양한 공격 유형 고루 포함)', fontweight='bold')
axes[0].legend()

# 상위 3개 방법의 공격 유형 분포 파이차트
top3 = ordered[:3]
from itertools import cycle
for i, (m, ax) in enumerate(zip(top3, [None,None,None])):
    pass

# 대신: 상위/하위 방법의 공격 분포 비교 (stacked bar)
aiso_dist = coverage.get('AISO', {}).get('dist', pd.Series())
pso_dist  = coverage.get('PSO',  {}).get('dist', pd.Series())
all_types = sorted(set(list(aiso_dist.index)+list(pso_dist.index)))
x = np.arange(len(all_types))
w = 0.35
aiso_vals = [aiso_dist.get(t, 0) for t in all_types]
pso_vals  = [pso_dist.get(t, 0)  for t in all_types]
aiso_p = np.array(aiso_vals)/sum(aiso_vals+[1e-9])
pso_p  = np.array(pso_vals)/sum(pso_vals+[1e-9])
axes[1].bar(x-w/2, aiso_p, w, label='AISO', color='#8B0000', alpha=0.8)
axes[1].bar(x+w/2, pso_p,  w, label='PSO',  color='blue',    alpha=0.8)
axes[1].set_xticks(x); axes[1].set_xticklabels(all_types, rotation=45, ha='right', fontsize=8)
axes[1].set_title('AISO vs PSO: 오버샘플 공격 유형 분포\n(AISO가 희귀 유형에 더 균등)', fontweight='bold')
axes[1].set_ylabel('비율'); axes[1].legend()

plt.tight_layout()
plt.savefig('cicids_minority_coverage.png', bbox_inches='tight', dpi=120)
plt.show()

Coverage 계산 중...
  Random             entropy=1.390  unique types=11
  K-Means            entropy=1.574  unique types=11
  Greedy             entropy=1.585  unique types=11
  Top-density        entropy=0.000  unique types=2
  ACO                entropy=1.387  unique types=11
  PSO                entropy=1.336  unique types=11
  SPSO               entropy=1.361  unique types=11
  CDE                entropy=1.345  unique types=11
  AISO               entropy=1.410  unique types=11
  RandomOver         entropy=1.390
